# C/GMRES

Continuation / GMRES という非線形制御問題の制約条件を考慮した制御入力をPMP条件をもとに求めるものである。

PMP条件を以下に示す。Hamilltonianを用いると表記が簡単になるが、まずは理解のためこのように記述する。また、最初であるため、時間 $t$ も記述する。

状態方程式

$$
\boxed{
\dot{x}(t)= f(x(t), u(t), t)
}
$$

随伴方程式

$$
\boxed{
\dot {\lambda}(t) = - L_x(x(t),u(t), t) - {f_x(x(t),u(t), t)}^T \lambda(t)
}
$$

停留条件
$$
\boxed{
L_u(x(t),u(t), t) + {f_u(x(t),u(t), t)}^T \lambda(t) = 0
}
$$

終端条件

$$
\boxed{
\lambda(T) = {\Phi_x}(x(T))
}
$$


## C/GMRESの流れ

PMPは以下の最適化制御問題を上記の4つの方程式に変換して、時間 $[0,T]$ すべてにおいて、これらを満たす $x(t), u(t), \lambda(t)$ を求める問題である。 

$$
\min\limits_{x(t), u(t)} \space J = \Phi(x(T)) + \int_0^T L (x,u,t) dt \\
\text{subject to} \space \dot{x} = f(x,u,t), \space x(0) = x_0
$$

ここでは有限の時間区間 $[0,T]$ における最適制御問題にPMPを適用している。
一方、実際の制御ではずっと制御ループを回し続けるため、実時間 $t$ が進み続ける。<br>
C/GMRESでは以下の図のように、各時刻 $t_k$ において、現在時刻から一定時間先までの予測ホライゾンを設定する。<br>
予測ホライゾン内の時間を予測時間 $\tau$ として、 $\tau \in [0,T]$ においてPMP条件を適用し、最適な制御入力の時系列を求める。

<img src="images/c-gmres_time_images.png" style="width:60%;">

実時間 $t$ と予測時間 $\tau$ は異なり、$\tau$ 方向の計算中に実時間 $t$ が進んでいることを意味するものではない。 

C/GMRESの計算は以下のように進めていく。状態 $x$ と随伴変数 $\lambda$ は多変数のベクトルも考慮できるが、ここでは理解のため $x$, $\lambda$ は1変数とする。



### Step 0

以下のように 、初期の制御入力 $U(0)$ を 予測ホライゾンの時間区間 $[0, T]$ で離散時間で設定する。

$$
\begin{aligned}
U(0) &= [u_0(0), u_1(0), \cdots, u_{N-1}(0)] \\
\end{aligned}
$$

以降のStepでの制御入力の記述は、制御ループの繰り返しを記述するため、$U(t_k)$ の表記を行う。

### Step 1

状態方程式 $\dot{x}(\tau)= f(x(\tau), u(\tau), t_k + \tau)$ をオイラー法で時間区間 $[0,T]$ まで初期状態 $x[0] = x(t_k)$ から $x[N]$ の時系列を求める。$h$ はステップ幅である。

$$
x[n+1] = x[n] + h \ f(x[n], u_n(t_k), t_k + \tau[n]) , \space n = 0,1, \cdots , N-1
$$

ここで求まる $x$ の時系列はものである。

$$
x[0], x[1] , \cdots , x[N]
$$

### Step 2

終端条件から、随伴変数の最終値 $\lambda[N]$ をStep1で求めた状態 $x[N]$ を用いて求める。

$$
\lambda[N] = {\Phi_x}(x[N])
$$

### Step 3

随伴方程式 $\dot {\lambda}(\tau) = - L_x(x(\tau),u(\tau), t_k + \tau) - {f_x(x(\tau),u(\tau), t_k + \tau)}^T \lambda(\tau)$ を時間を逆方向に進める $N-1$ から $1$ まで計算して、随伴変数 $\lambda$ の時系列を求める。$x$は1変数であるため本来は $f_x$ の転置は必要ないが、記述として残している。

$$
\lambda[n] = \lambda[n+1] + h L_x(x[n], u_n, t[n]) + \left( h f_x(x[n], u_n, t[n]) \right)^T\lambda[n+1]  , \quad n = N-1 , N-2, \cdots , 1
$$

この計算式は以下の考え方に基づいている。

----------------------------------------------------------------------------

評価関数 $J$ を離散化する。積分区間は区分求積法としている。

$$
J = \Phi(x[N]) + \sum_{n=0}^{N-1} \ h \ L(x[n], u_n, t[n])
$$

制約条件はオイラー法を用いた離散化により、以下のようになる。

$$
x[n+1] - x[n] - h \ f(x[n], u_n, t[n]) = 0
$$

PMPの拡大評価関数に合わせて制約条件を組み込むと、離散化した拡大評価関数 $\bar{J}$ は以下のようになる。

$$
\bar{J} = \Phi(x[N]) + \sum_{n=0}^{N-1} \left( \ h \ L(x[n], u_n, t[n]) + \lambda[n+1]^T(x[n] + h \ f(x[n], u_n, t[n]) - x[n+1] ) \right)
$$

PMPの随伴方程式は $x$ による偏微分から求めたため、 $\bar{J}$ を $L$と $f$ の引数である $x[n]$ で偏微分する。

$$
\frac{\partial \bar{J}}{\partial x[n]} = 0
$$

積分内部をいくつか展開する。

$$
\begin{aligned}
h L (x[0], u_0, t[0]) + \lambda[1]^T(x[0] + h f(x[0],u_0, t[0]) - x[1]) \quad , n=0 \\
h L (x[1], u_1, t[1]) + \lambda[2]^T(x[1] + h f(x[1],u_1, t[1]) - x[2]) \quad , n=1 \\
h L (x[2], u_2, t[2]) + \lambda[3]^T(x[2] + h f(x[2],u_2, t[2]) - x[3]) \quad , n=2 \\
\end{aligned}
$$

これを $x[n]$ についてまとめると、以下のように $\lambda[n]^T x[n]$ が一つずつ移っていく。 

$$
\begin{array}{l}
h L (x[0], u_0, t[0]) + \lambda[1]^T (x[0] + h f(x[0], u_0, t[0])) + \\
h L (x[1], u_1, t[1]) + \lambda[2]^T (x[1] + h f(x[1], u_1, t[1])) - \lambda[1]^T \ x[1] + \\
h L (x[2], u_2, t[2]) + \lambda[3]^T (x[2] + h f(x[2], u_2, t[2])) - \lambda[2]^T \ x[2] + \\
\vdots \\
h L (x[N-1], u_{N-1}, t[N-1]) + \lambda[N]^T (x[N-1] + h f(x[N-1], u_{N-1}, t[N-1])) - \lambda[N-1]^T \ x[N-1]
\end{array}
$$

よって、$ 1 \le n \le N-1$ の区間では以下のようになる。

$$
h L (x[n], u_n, t[n]) + \lambda[n+1]^T (x[n] + h f(x[n], u_n, t[n])) - \lambda[n]^T \ x[n]
$$

これを $x[n]$ で変微分し "ゼロ" とする。

$$
h L_x(x[n], u_n, t[n]) + \lambda[n+1] + \left( h f_x(x[n], u_n, t[n]) \right)^T \lambda[n+1]  - \lambda[n] = 0
$$

ここから、$\lambda$ の 時系列を次のように計算できる。

$$
\lambda[n] = \lambda[n+1] + h L_x(x[n], u_n, t[n]) + \left( h f_x(x[n], u_n, t[n]) \right)^T\lambda[n+1]  , \quad n = N-1 , N-2, \cdots , 1
$$

----------------------------------------------------------------------------

これにより、次の $\lambda$ の時系列が求まる。

$$
\lambda[N], \lambda[N-1] , \cdots , \lambda[1]
$$

### Step 4

制御入力 $U(t_k)$ を仮定し、 Step 1から Step 3 で 状態変数 $x[n]$ 、随伴変数 $\lambda[n]$、の時系列を予測ホライゾン区間を求めた。<br>
Step 4 ではこれらを使い、停留条件の式 $L_u(x(\tau),u(\tau), t_k + \tau) + {f_u(x(\tau),u(\tau), t_k + \tau)}^T \lambda(\tau) = 0$ を 具体的な数値の縦のベクトルを形成する。これを $F$ として定義する。

$$
F(U(t_k), x(t_k), t_k) =
\begin{bmatrix}
L_u(x[0],u_0(t_k), \tau[0]) + {f_u(x[0],u_0(t_k), t_k + \tau[0])}^T \lambda[1] \\
L_u(x[1],u_1(t_k), \tau[1]) + {f_u(x[1],u_1(t_k), t_k + \tau[1])}^T \lambda[2] \\
\vdots \\
L_u(x[N-1],u_{N-1}(t_k), \tau[N-1]) + {f_u(x[N-1],u_{N-1}(t_k), t_k + \tau[N-1])}^T \lambda[N]
\end{bmatrix} = 0
$$

この $F(U(t_k), x(t_k), t_k) = 0 $ は$t_k$から始まる予測ホライゾン上のすべての離散点で、停留条件を満たすことを意味している。




### Step 5

Step 4 では $t_k$ における制御入力 $U(t_k)$から状態変数 $x[n]$ 、随伴変数 $\lambda[n]$ の時系列を求め、予測ホライゾン上の停留条件をまとめた $F(U(t_k), x(t_k), t_k)$ を求めた。<br>
理想的には $F=0$ を満たす制御入力を次の時刻 $t_{k+1}$ においても追従したい。<br>
まずは理想的に $F=0$が成立しているとして、その状態を次の時刻でも維持するために、時間変化がゼロであるとして次の関係を導く。

$$
\frac{d \ F ( U(t_k), x(t_k), t_k)}{d \ t} = 0
$$

上式は $t$ による全微分であるため、チェインルールを適用すると以下となる。記述の簡略化のため、$t_k=t$とする。

$$
\begin{aligned}
\frac{d \ F ( U(t), x(t), t)}{d \ t} &= \frac{\partial \ F }{\partial \ U} \frac{ d U}{dt} + \frac{\partial \ F }{\partial \ x} \frac{ d x}{dt} + \frac{\partial \ F }{\partial \ t} \\
&= F_U \dot{U} + F_x \dot{x} + F_t = 0
\end{aligned}
$$

- $F_U \ \dot{U}$ : $U$ が実時間方向に変化することによる $F$ の変化率
- $F_x \ \dot{x}$ : $x$ が実時間方向に変化することによる $F$ の変化率
- $F_t$ : 予測ホライゾンの起点 $t_k$ が進むことによる、実時間への明示的既存を通した $F$ の変化率

これより、以下の式が現れる。

$$
F_U \dot{U} = -\left(F_x \dot{x} + F_t \right)
$$

$U$はベクトルであるため、その偏微分 $F_U$ はヤコビアン(行列)になる。また、右辺はベクトルになるため、この式は $A z = b$ のように、一次連立方程式であり、GMRESで解くことが出来る。<br>
そして、これを解くことにより、$\dot{U}$が求まるため、次の制御入力は以下のように求めることが出来る。

$$
U(t_{k+1}) = U(t_k) + \Delta x \ \dot{U}(t_k)
$$

ここまでで停留条件を満たしつつ、制御入力 $U$ を計算していく方法を説明した。


### $\dot{U}$ の計算

以下の式に含まれる $F_U, F_x , F_t$ を解析的に求めることもできるが、計算が膨大になる。そこで C/GMRESでは有限差分で右辺と左辺のそれぞれを求めていく。

$$
F_U \dot{U} = -\left(F_x \dot{x} + F_t \right)
$$

まず右辺の計算方法を考える。

#### 右辺を構成する $F_x \dot{x} + F_t$ 

$U$ は固定して、$x,t$ の少し先を以下のようにする。

$$
\Delta x = \varepsilon \dot{x} , \quad \Delta t = \varepsilon
$$

すると、$F(U, x+\Delta x , t + \Delta t)$ の多変数のテイラー展開を1次変化までとすると、以下のようになる。

$$
F(U, x+\Delta x , t + \Delta t) \simeq F(U, x, t) + F_x \Delta x + F_t \Delta t
$$

ここで、$\Delta x = \varepsilon \dot{x} , \ \Delta t = \varepsilon$ を代入すると以下のようになる。

$$
\begin{aligned}
F(U, x+ \varepsilon \dot{x}, t + \varepsilon) &\simeq F(U, x, t) + F_x \varepsilon \dot{x}  + F_t \varepsilon \\
F(U, x+ \varepsilon \dot{x}, t + \varepsilon) &\simeq F(U, x, t) + \varepsilon  \left(F_x \dot{x}  + F_t \right) \\
F(U, x+ \varepsilon \dot{x}, t + \varepsilon) - F(U, x, t) &\simeq \varepsilon  \left(F_x \dot{x}  + F_t \right) \\
\end{aligned}
$$

これより、以下の有限差分で近似できる。

$$
F_x \dot{x}  + F_t \simeq \frac{F(U, x+ \varepsilon \dot{x}, t + \varepsilon) - F(U, x, t) }{\varepsilon} \\
$$

C/GMRESの記述に合わせると、以下のようになる。

$$
\boxed{
F_x \dot{x}  + F_t \simeq \frac{F(U(t_k), x(t_k)+ \varepsilon \dot{x}(t_k), t_k + \varepsilon) - F(U(t_k), x(t_k), t_k) }{\varepsilon} \\
}
$$

これは、$F_x \dot{x}  + F_t$ を求めるために、Step1 から Step 4を $x$ と $t$ に関するものを設定しなおして $F$を再計算する。

##### $x(t_k) + \varepsilon \dot{x}(t_k)$ の影響

Step1 では 初期値 $x[0] = x(t_k)$として 状態の時系列を求めたが、これを $x_\varepsilon[0] = x(t_k) + \varepsilon \dot{x}(t_k)$ として計算していく。ここで $x(t_k)$ は実時間が $t_k$ 時の制御対象の状態である。

これにより、$x_\varepsilon[0], x_\varepsilon[1], \cdots , x_\varepsilon[N]$ の時系列をStep1で再計算する。

ここで、$\dot{x}(t_k)$ は以下のように状態方程式から求めることが出来る。

$$
\dot{x}(t_k) = f(x(t_k), u_0(t_k) , t_k)
$$

例えば制御対象が簡単に質点モデルである場合、状態を$[x(t_k), v(t_k)]$とすると、状態方程式は以下となる。

$$
\frac{d}{dt}
\begin{bmatrix}
x(t_k) \\ v(t_k)
\end{bmatrix} =
\begin{bmatrix}
v(t_k) \\ u_0(t_k)/m
\end{bmatrix}
$$

よって、$t_k$時刻の $v(t_k)$ と制御入力 $u_0(t_k)$があれば計算できる。 状態はすべて観測可能と考えてC/GMRESの計算を行うが、観測できなればKalman Filter等の状態推定を行う必要がある。しかし、これはC/GMRESとは異なるトピックであるため除く。また、$u_0(t_k)$は予測ホライゾン上の最初の制御入力の値である。

##### $t_k + \varepsilon$ の影響

$t_k + \varepsilon$ は $F$ に含まれる実時間 $t$ に明示的に依存する項を、微小時間 $\varepsilon$ だけ進めて評価することを意味する。

例えば目標値 $x_{ref}(t)$ が時間変化する場合、ランニングコスト $L$ を考えた場合、連続時間では以下となる。

$$
L(x,u,t) = \frac{1}{2} q ( x - x_{ref}(t))^2 + \frac{1}{2} r \ u^2
$$ 

ここで目標値 を $x_{ref}(t) = \sin(t)$ とした場合、予測ホライゾン上の$n$番目の時刻は $t = t_k + \tau[n]$ になるため、$L$は以下となる。

$$
L(x[n] , u_n, t_k + \tau[n]) =  \frac{1}{2} q ( x[n] - \sin(t_k + \tau[n]))^2 + \frac{1}{2} r \ u_n^2
$$

これを $\varepsilon$ 進めて考えるため以下のようになる。

$$
L(x_\varepsilon[n], u_n, t_k + \varepsilon + \tau[n]) = \frac{1}{2} q ( x_\varepsilon[n] - \sin(t_k + \varepsilon + \tau[n]))^2 + \frac{1}{2} r \ u_n^2
$$

これが $t_k + \varepsilon$ による $F_t$ に現れる一つの影響である。<br>
これ以外にも終端コストや状態方程式でも明示的に実時間 $t$ に依存する場合は、$t_k + \varepsilon$ を基準として再評価を行う。

一方、$F$ を構成する状態方程式、ランニングコスト、終端コスト、目標値などに実時間 $t$ への明示的な依存がなければ $F_t = 0 $となる。

#### 左辺を構成する $F_U \dot{U}$

式を再掲する。

$$
F_U \dot{U} = -\left(F_x \dot{x} + F_t \right)
$$

左辺の$F_U$は以下のように計算される

$x,t$ を固定して $U$の少し先を次のようにする。

$$
\Delta U = \varepsilon \dot{U}
$$

すると $F(U + \Delta U , x, t)$ の多変数のテイラー展開の1次変化までとすると、

$$
F(U + \Delta U , x, t) \simeq F(U , x, t) + F_U \Delta U 
$$

ここで $\Delta U$ を代入すると、

$$
\begin{aligned}
F(U + \varepsilon \dot{U}, x, t) \simeq& F(U , x, t) + F_U \ \varepsilon \dot{U} \\
F(U + \varepsilon \dot{U}, x, t) - F(U , x, t) \simeq& F_U \ \varepsilon \dot{U} \\
F_U \ \dot{U} \simeq& \frac{F(U + \varepsilon \dot{U}, x, t) - F(U , x, t)}{\varepsilon}
\end{aligned}
$$

しかし、求めたいものは $\dot{U}$ であり、
